In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "geopandas",
#     "lonboard",
#     "matplotlib",
#     "palettable",
#     "pandas",
#     "pyarrow",
#     "shapely",
# ]
# ///

import geopandas as gpd
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
from palettable.colorbrewer.sequential import YlGn_9
from shapely.geometry import Point

from lonboard import Map, PointCloudLayer
from lonboard.basemap import CartoStyle, MaplibreBasemap
from lonboard.colormap import apply_continuous_cmap
from lonboard.view_state import MapViewState

# 1. GENERATE RICH 3D FOREST CANOPY LIDAR DATA
np.random.seed(42)
num_trees = 150
points_per_tree = 80

# Plot geographic center (Redwood National Park area)
center_lon, center_lat = -124.004, 41.213

lons, lats, heights, intensities = [], [], [], []
return_numbers, classifications, tree_ids = [], [], []

for tree_idx in range(num_trees):
    # Base position and max height for each tree
    tree_lon = center_lon + np.random.normal(0, 0.002)
    tree_lat = center_lat + np.random.normal(0, 0.002)
    max_tree_height = np.random.uniform(15, 65)  # Tree height in meters
    tree_id_label = f"TREE_{tree_idx + 1:03d}"

    for _ in range(points_per_tree):
        # 3D elevation Z
        z = np.random.beta(2, 1) * max_tree_height
        
        # Spatial dispersion forming tree crown
        crown_radius = (z / max_tree_height) * np.random.uniform(0.0001, 0.0003)
        x = tree_lon + np.random.normal(0, crown_radius)
        y = tree_lat + np.random.normal(0, crown_radius)
        
        # Attribute generation
        if z < 1.5:
            classification = "Ground"
            return_num = 3  # Last return
            intensity = np.random.uniform(0.05, 0.25)
        elif z < 10.0:
            classification = "Medium Vegetation"
            return_num = 2  # Intermediate branch return
            intensity = np.random.uniform(0.30, 0.60)
        else:
            classification = "High Vegetation"
            return_num = 1  # First canopy return
            intensity = np.random.uniform(0.60, 1.00)

        lons.append(x)
        lats.append(y)
        heights.append(z)
        intensities.append(intensity)
        return_numbers.append(return_num)
        classifications.append(classification)
        tree_ids.append(tree_id_label)

# 2. CONSTRUCT 3D GEOPANDAS GEODATAFRAME
# Points constructed as 3D Shapely Geometries: Point(longitude, latitude, elevation_z)
geometry_3d = [
    Point(x, y, z)
    for x, y, z in zip(lons, lats, heights, strict=True)
]

gdf_forest = gpd.GeoDataFrame(
    {
        "canopy_height_m": heights,
        "lidar_intensity": intensities,
        "return_number": return_numbers,
        "classification": classifications,
        "tree_id": tree_ids,
    },
    geometry=geometry_3d,
    crs="EPSG:4326",
)

# 3. CONTINUOUS COLOR MAPPING (Mapped to Canopy Height)
color_scale = Normalize(
    vmin=gdf_forest["canopy_height_m"].min(),
    vmax=gdf_forest["canopy_height_m"].max(),
)
fill_colors = apply_continuous_cmap(
    color_scale(gdf_forest["canopy_height_m"]), YlGn_9, alpha=0.9
)

# 4. INSTANTIATE PointCloudLayer (Lonboard API Pattern)
# 'pickable=True' automatically populates all GeoDataFrame columns into hover tooltips
point_cloud_layer = PointCloudLayer.from_geopandas(
    gdf_forest,
    get_color=fill_colors,
    point_size=3.5,            # Point size in pixels
    size_units="pixels",
    pickable=True,
)

# 5. RENDER MAP WITH 3D CAMERA TILT
map_ = Map(
    point_cloud_layer,
    basemap=MaplibreBasemap(style=CartoStyle.DarkMatter),
    view_state=MapViewState(
        longitude=center_lon,
        latitude=center_lat,
        zoom=16,
        pitch=60,              # 60-degree tilt reveals vertical 3D forest canopy structure
        bearing=30,
    ),
    height=700,
    show_tooltip=True,
)

map_